In [591]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (StratifiedKFold, RepeatedStratifiedKFold,
                                     cross_val_score, cross_val_predict, GridSearchCV, train_test_split)
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

train_df = pd.read_csv('data/train.csv', index_col=[0])
test_df = pd.read_csv('data/test.csv', index_col=[0])

## treino

In [592]:
train_df.head()

,pelvic_incidence,pelvic_tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,degree_spondylolisthesis,class
0,39.056951,10.060991,25.015378,28.995960,114.405425,4.564259,Hernia
1,68.832021,22.218482,50.092194,46.613539,105.985135,-3.530317,Hernia
2,49.712859,9.652075,28.317406,40.060784,108.168725,7.918501,Hernia
3,40.250200,13.921907,25.124950,26.328293,130.327871,2.230652,Hernia
4,53.432928,15.864336,37.165934,37.568592,120.567523,5.988551,Hernia


In [593]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 217 entries, 0 to 216
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   pelvic_incidence          217 non-null    float64
 1   pelvic_tilt               217 non-null    float64
 2   lumbar_lordosis_angle     217 non-null    float64
 3   sacral_slope              217 non-null    float64
 4   pelvic_radius             217 non-null    float64
 5   degree_spondylolisthesis  217 non-null    float64
 6   class                     217 non-null    str    
dtypes: float64(6), str(1)
memory usage: 12.0 KB


In [594]:
train_df.describe()

,pelvic_incidence,pelvic_tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,degree_spondylolisthesis
count,217.000000,217.000000,217.000000,217.000000,217.000000,217.000000
mean,60.458343,17.541355,52.660081,42.916988,118.073520,27.715448
std,18.045677,10.786425,19.226267,13.631236,13.920744,40.965137
min,26.147921,-6.554948,14.000000,13.516568,70.082575,-11.058179
25%,46.236399,9.976664,36.329137,33.366366,110.660701,1.517203
50%,57.145851,16.417462,51.000000,42.447102,118.151531,13.683047
75%,74.469082,22.432950,63.232302,51.766175,126.398188,49.159426
max,129.834041,49.431864,125.742385,121.429566,163.071041,418.543082


contagem de classes

In [595]:
train_df.iloc[:,-1].value_counts()

class
Spondylolisthesis    105
Normal                70
Hernia                42
Name: count, dtype: int64

ta desbalanceado. fato que precisa de estratificação

"A Kernel Density Estimation (KDE) plot is a type of plot that visualizes the estimated probability density function of a continuous variable. Unlike a histogram, which displays data using discrete bars grouped into intervals, a KDE plot represents the distribution as a smooth, continuous curve based on all data points." https://codefinity.com/courses/v2/61928667-6699-4af9-a146-b70e73fce3c5/d6354309-3168-44c6-b630-26cf849bd32f/f2dcae95-a3e5-4751-8730-870f0050101b

Eu tenho claramente variaveis numericas e continuas (pode asusmir inf valores dentro de um intervalo)

In [596]:
train_df = train_df[train_df['degree_spondylolisthesis'] < 400]

X são as features (6 atributos) e y eh a classe que sera prevista

In [597]:
X = train_df.drop(columns=['class'])
y = train_df['class']

In [598]:
X

,pelvic_incidence,pelvic_tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,degree_spondylolisthesis
0,39.056951,10.060991,25.015378,28.995960,114.405425,4.564259
1,68.832021,22.218482,50.092194,46.613539,105.985135,-3.530317
2,49.712859,9.652075,28.317406,40.060784,108.168725,7.918501
3,40.250200,13.921907,25.124950,26.328293,130.327871,2.230652
4,53.432928,15.864336,37.165934,37.568592,120.567523,5.988551
...,...,...,...,...,...,...
212,54.600316,21.488974,29.360216,33.111342,118.343321,-1.471067
213,34.382299,2.062683,32.390820,32.319617,128.300199,-3.365516
214,45.075450,12.306951,44.583177,32.768499,147.894637,-8.941709
215,53.936748,20.721496,29.220534,33.215251,114.365845,-0.421010


In [599]:
y

0      Hernia
1      Hernia
2      Hernia
3      Hernia
4      Hernia
        ...  
212    Normal
213    Normal
214    Normal
215    Normal
216    Normal
Name: class, Length: 216, dtype: str

In [600]:
# mantend a msm ordem d colunas do treino 
X_test = test_df[X.columns]

In [601]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

cross_val = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "decision tree": make_pipeline(RobustScaler(), DecisionTreeClassifier(max_depth=40, random_state=42)),
    "knn(euclidian)": make_pipeline(RobustScaler(), KNeighborsClassifier(n_neighbors=10, metric='euclidean', weights='distance')),
}

In [602]:
results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cross_val, scoring='f1_macro', n_jobs=-1)
    results[name] = scores
    print(f"name={name:32s}, mean={scores.mean():.4f}, std={scores.std():.4f}")

name=decision tree                   , mean=0.7451, std=0.0597
name=knn(euclidian)                  , mean=0.7793, std=0.0768


# Decision Tree

In [603]:
params_grid = {
    'decisiontreeclassifier__max_depth': [2, 3, 4, 5, 6, 8, 10, 11, 12, 13, 14, 15],
    'decisiontreeclassifier__min_samples_leaf': [1, 3, 5, 10],
    'decisiontreeclassifier__ccp_alpha': [0.0, 0.01, 0.02, 0.05],
}

In [604]:
for scaler in [RobustScaler(), StandardScaler(), MinMaxScaler()]:
    pipe = make_pipeline(scaler, DecisionTreeClassifier(random_state=42))
    gs = GridSearchCV(estimator=pipe, param_grid=params_grid, cv=cross_val, scoring='f1_macro', n_jobs=-1)
    gs.fit(X, y)

    print("best params:", gs.best_params_)
    print('best acc:', gs.best_score_)


best params: {'decisiontreeclassifier__ccp_alpha': 0.0, 'decisiontreeclassifier__max_depth': 2, 'decisiontreeclassifier__min_samples_leaf': 1}
best acc: 0.8001430623223571
best params: {'decisiontreeclassifier__ccp_alpha': 0.0, 'decisiontreeclassifier__max_depth': 2, 'decisiontreeclassifier__min_samples_leaf': 1}
best acc: 0.8001430623223571
best params: {'decisiontreeclassifier__ccp_alpha': 0.0, 'decisiontreeclassifier__max_depth': 2, 'decisiontreeclassifier__min_samples_leaf': 1}
best acc: 0.8001430623223571


## KNN

In [605]:
knn_results = []
print("KNN")
for n in [2, 3, 4, 5, 10, 20, 30, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]:
    for metric in ['minkowski', 'euclidean', 'manhattan', 'chebyshev']:
        for scaler in [RobustScaler(), StandardScaler()]:
            pipe = make_pipeline(scaler, KNeighborsClassifier(n_neighbors=n, metric=metric, weights='distance'))
            score = cross_val_score(pipe, X, y, cv=cross_val, scoring="f1_macro", n_jobs=-1)
            print(f"knn: n={str(n):>9}, metric={metric}, score_mean={score.mean():.4f}, score_std={score.std():.4f}")
            knn_results.append({
                "n": n,
                "scaler": scaler,
                "metric": metric,
                "score_mean": score.mean(),
                "score_std": score.std(),
            })

KNN
knn: n=        2, metric=minkowski, score_mean=0.7950, score_std=0.0404
knn: n=        2, metric=minkowski, score_mean=0.8124, score_std=0.0192
knn: n=        2, metric=euclidean, score_mean=0.7950, score_std=0.0404
knn: n=        2, metric=euclidean, score_mean=0.8124, score_std=0.0192
knn: n=        2, metric=manhattan, score_mean=0.8178, score_std=0.0441
knn: n=        2, metric=manhattan, score_mean=0.8351, score_std=0.0402
knn: n=        2, metric=chebyshev, score_mean=0.7739, score_std=0.0344
knn: n=        2, metric=chebyshev, score_mean=0.7755, score_std=0.0417
knn: n=        3, metric=minkowski, score_mean=0.7654, score_std=0.0719
knn: n=        3, metric=minkowski, score_mean=0.7783, score_std=0.0697
knn: n=        3, metric=euclidean, score_mean=0.7654, score_std=0.0719
knn: n=        3, metric=euclidean, score_mean=0.7783, score_std=0.0697
knn: n=        3, metric=manhattan, score_mean=0.7820, score_std=0.0763
knn: n=        3, metric=manhattan, score_mean=0.7976, score

In [606]:
best_knn = max(knn_results, key=lambda c: (c['score_mean'], -c['score_std']))
best_knn

{'n': 4,
 'scaler': StandardScaler(),
 'metric': 'manhattan',
 'score_mean': np.float64(0.8529582377090577),
 'score_std': np.float64(0.03817736187842672)}

In [607]:
# final = DecisionTreeClassifier(max_depth=3, random_state=42)
# final.fit(X, y)
final = make_pipeline(RobustScaler(), KNeighborsClassifier(n_neighbors=best_knn['n'], metric=best_knn['metric'], weights='distance'))
final.fit(X, y)
preds = final.predict(X_test)
preds

array(['Spondylolisthesis', 'Spondylolisthesis', 'Hernia', 'Hernia',
       'Normal', 'Spondylolisthesis', 'Normal', 'Spondylolisthesis',
       'Hernia', 'Hernia', 'Hernia', 'Normal', 'Spondylolisthesis',
       'Normal', 'Spondylolisthesis', 'Spondylolisthesis', 'Normal',
       'Hernia', 'Normal', 'Spondylolisthesis', 'Normal',
       'Spondylolisthesis', 'Normal', 'Hernia', 'Spondylolisthesis',
       'Normal', 'Normal', 'Spondylolisthesis', 'Normal', 'Hernia',
       'Spondylolisthesis', 'Hernia', 'Normal', 'Spondylolisthesis',
       'Normal', 'Normal', 'Normal', 'Normal', 'Hernia',
       'Spondylolisthesis', 'Spondylolisthesis', 'Normal', 'Hernia',
       'Normal', 'Spondylolisthesis', 'Spondylolisthesis', 'Hernia',
       'Spondylolisthesis', 'Spondylolisthesis', 'Spondylolisthesis',
       'Spondylolisthesis', 'Normal', 'Spondylolisthesis',
       'Spondylolisthesis', 'Normal', 'Hernia', 'Normal',
       'Spondylolisthesis', 'Hernia', 'Hernia', 'Spondylolisthesis',
       'Sp

### Voting

In [608]:
simple_cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
repeated_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=42)

In [609]:
def evaluation(name, model, cv=repeated_cv, verbose=True):
    s = cross_val_score(model, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)
    if verbose:
        print(f'{name:44s} {s.mean():.4f}')
    return s.mean()

In [610]:
evaluation('baseline dummy', DummyClassifier(strategy='most_frequent'));

baseline dummy                               0.2167


In [611]:
# from sklearn.preprocessing import PowerTransformer
# from sklearn.ensemble import BaggingClassifier
# nb_simple = make_pipeline(GaussianNB())
# #knn_power = make_pipeline(PowerTransformer(), KNeighborsClassifier(n_neighbors=5))
# knn_power = make_pipeline(RobustScaler(), KNeighborsClassifier(n_neighbors=6, metric=best_knn['metric'], weights='distance'))
# #knn_otimizado = make_pipeline(RobustScaler(), KNeighborsClassifier(n_neighbors=best_knn['n'], metric=best_knn['metric'], weights='distance'))
# dt_opt = DecisionTreeClassifier(max_depth=4, min_samples_leaf=1,ccp_alpha=0.0, random_state=42, class_weight='balanced')
# #best params: {'decisiontreeclassifier__ccp_alpha': 0.0, 'decisiontreeclassifier__max_depth': 2, 'decisiontreeclassifier__min_samples_leaf': 1}
# dt_bagged = BaggingClassifier(
#     estimator=dt_opt, 
#     n_estimators=100, 
#     random_state=42
# )
# # final = VotingClassifier(
# #     estimators=[
# #         #('knn_power_2', knn_power_2),
# #         ('knn_power', knn_power),
# #         #('knn', knn_otimizado), 
# #         ('nb', nb_simple), 
# #         ('dt', dt_opt)
# #     ], 
# #     voting='hard'
# # )

# final = VotingClassifier(
#     estimators=[
#         #('knn_power_2', knn_power_2),
#         ('knn_power', knn_power),
#         #('knn', knn_otimizado), 
#         ('nb', nb_simple), 
#         ('dt', dt_bagged)
#     ], 
#     voting='soft'
# )

# results = {}
# for nome, m in {'knn power': knn_power, 'nb Default': nb_simple, 'dt bootstrap': dt_bagged, 'FINAL VOTING': final}.items():
#     results[nome] = [cross_val_score(m, X, y, n_jobs=-1, scoring='f1_macro',
#                     cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=s)).mean()
#                     for s in [7, 42, 59, 89, 101, 999, 10000]]

# print("acc mean 7 seeds 100 folds")
# print(pd.DataFrame(results, index=['seed=7', 'seed=42', 'seed=59', 'seed=89', 'seed=101', 'seed=999', 'seed=10000']).T.round(4).to_string())

In [612]:
# from sklearn.preprocessing import PowerTransformer

# knn_final = make_pipeline(
#     StandardScaler(), 
#     KNeighborsClassifier(n_neighbors=4, metric='euclidean', weights='distance')
# )

# nb_final = make_pipeline(
#     PowerTransformer(), 
#     GaussianNB()
# )

# dt_final = DecisionTreeClassifier(max_depth=2, random_state=42)

# final = VotingClassifier(
#     estimators=[
#         ('knn', knn_final), 
#         ('nb', nb_final), 
#         ('dt', dt_final)
#     ], 
#     voting='soft'
# )

# final.fit(X, y)
# predicoes = final.predict(X_test)


# print("\n--- DISTRIBUIÇÃO ALVO: SL=45, Normal=30, Hernia=18 ---")
# print("Sua nova distribuição submetida:")
# print(pd.Series(predicoes).value_counts())

In [613]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [614]:
# final.fit(X, y)
final = KNeighborsClassifier(n_neighbors=7, weights='distance')
final.fit(X_train, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",7
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'distance'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[object](3,)","['Hernia','Normal','Spondylolisthesis']"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


In [615]:
preds = final.predict(X_test)
preds

array(['Spondylolisthesis', 'Spondylolisthesis', 'Hernia',
       'Spondylolisthesis', 'Hernia', 'Normal', 'Spondylolisthesis',
       'Normal', 'Spondylolisthesis', 'Normal', 'Normal', 'Hernia',
       'Spondylolisthesis', 'Hernia', 'Spondylolisthesis',
       'Spondylolisthesis', 'Spondylolisthesis', 'Normal', 'Normal',
       'Hernia', 'Normal', 'Normal', 'Normal', 'Spondylolisthesis',
       'Hernia', 'Hernia', 'Hernia', 'Spondylolisthesis', 'Normal',
       'Spondylolisthesis', 'Normal', 'Hernia', 'Spondylolisthesis',
       'Normal', 'Hernia', 'Spondylolisthesis', 'Spondylolisthesis',
       'Spondylolisthesis', 'Hernia', 'Normal', 'Spondylolisthesis',
       'Hernia', 'Spondylolisthesis', 'Spondylolisthesis'], dtype=object)

In [616]:
print('Accuracy:', accuracy_score(y_test, preds))
print('F1:', f1_score(y_test, preds, average='macro'))
print('Precision:', precision_score(y_test, preds, average='macro'))
print('Recall:', recall_score(y_test, preds, average='macro'))

Accuracy: 0.8863636363636364
F1: 0.8739858906525573
Precision: 0.8653846153846154
Recall: 0.8968253968253967


In [617]:
final_pred = final.predict(test_df)

In [618]:
def create_submission_file(predictions, test_df, submission_file_name="submission.csv"):
    submission_df = pd.DataFrame({"id": test_df.index, "Target": predictions})
    submission_df.to_csv(submission_file_name, index=False)
    print(f"Submission file '{submission_file_name}' created successfully.")


In [619]:
create_submission_file(final_pred, test_df)

Submission file 'submission.csv' created successfully.


In [620]:
#create_submission_file(preds, test_df, submission_file_name="submission_knn.csv")